# Fine-tuning Llama 3.2 3B para Text-to-SQL con LoRA usando Dataset CSV

Este notebook entrena un modelo especializado en generar consultas SQL usando **LoRA** sobre **Llama 3.2 3B** con datos desde CSV.

## ¿Qué haremos?
- Cargar Llama 3.2 3B Instruct
- Aplicar LoRA para fine-tuning eficiente
- Entrenar con datos de text-to-SQL desde CSV pre-limpiado
- Evaluar y guardar el modelo

## Requisitos:
- GPU con 12GB+ VRAM (Kaggle T4/P100)
- Cuenta Hugging Face
- Python 3.8+
- Dataset CSV estratificado

## 1. Instalación de Dependencias

In [ ]:
# Instalar todas las dependencias necesarias
!pip install transformers datasets accelerate peft bitsandbytes torch huggingface_hub pandas numpy trl
!pip install ipywidgets

print("✅ Dependencias instaladas")

## 2. Autenticación Hugging Face

In [ ]:
from huggingface_hub import login

# Login en Hugging Face (necesario para Llama)
login()

print("✅ Autenticado en Hugging Face")

## 3. Importación de Librerías

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

# LoRA y PEFT
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    TaskType
)

# Datasets y entrenamiento
from datasets import Dataset
from trl import SFTTrainer

print(f"✅ Librerías importadas")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💾 CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🎮 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Verificar recursos disponibles en Kaggle
print("🔍 VERIFICANDO RECURSOS KAGGLE")
print("=" * 40)

# Información del sistema
import psutil
print(f"💾 RAM Total: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"💾 RAM Disponible: {psutil.virtual_memory().available / 1024**3:.1f} GB")
print(f"🖥️ CPUs: {psutil.cpu_count()}")

# GPU info si está disponible
if torch.cuda.is_available():
    print(f"\n🎮 GPU DETECTADA:")
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"   GPU {i}: {gpu_name}")
        print(f"   VRAM: {gpu_memory:.1f} GB")
        
        # Memoria disponible
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        cached = torch.cuda.memory_reserved(i) / 1024**3
        print(f"   Allocated: {allocated:.1f} GB")
        print(f"   Cached: {cached:.1f} GB")
        print(f"   Free: {gpu_memory - allocated:.1f} GB")
else:
    print("\n⚠️ NO GPU DETECTADA - Se usará CPU")
    print("   Recomendación: Activar GPU en Kaggle Settings")

# Disk space
disk = psutil.disk_usage('/')
print(f"\n💿 Disk Space:")
print(f"   Total: {disk.total / 1024**3:.1f} GB")
print(f"   Free: {disk.free / 1024**3:.1f} GB")

print("\n✅ Verificación completada")

## 4. Configuración del Modelo y Entrenamiento

Configuración optimizada para **Llama 3.2 3B** y **text-to-SQL** con **Dataset CSV**:

In [ ]:
# ====== CONFIGURACIÓN PRINCIPAL ======
CONFIG = {
    # Modelo Llama 3.2 3B
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    
    # Dataset CSV
    "csv_file_path": "C:/Users/arasa/OneDrive - UTN - Santa Fe/Facultad/5° Año/Proyecto Final/proyecto-final-repo/proyecto_final/inputs/dataset_estratificado_10000.csv",
    "num_samples": 10000,  # Más muestras para aprovechar el modelo 3B
    
    # Directorios
    "output_dir": "../models/llama-sql-lora-csv",
    "logs_dir": "../logs",
    
    # Parámetros de entrenamiento optimizados para Kaggle
    "max_seq_length": 512,   # Secuencias más largas para 3B
    "batch_size": 1,         # Batch pequeño por memoria en Kaggle
    "gradient_accumulation": 8, # Simula batch_size = 8
    "learning_rate": 2e-4,   # LR más alto para 3B
    "num_epochs": 2,         # Más epochs para aprovechar el modelo
    "warmup_ratio": 0.05,    # Warmup más corto
    "save_steps": 50,        # Guardar más frecuente
    "eval_steps": 50,
    "eval_strategy": "steps",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "logging_steps": 10,
}

# ====== CONFIGURACIÓN LORA PARA LLAMA 3.2 3B ======
LORA_CONFIG = {
    "r": 16,                  # Rango más alto para modelo 3B
    "lora_alpha": 32,         # Alpha proporcional (2x rank)
    "lora_dropout": 0.05,     # Dropout menor para modelo más grande
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
    
    # Módulos específicos de Llama 3.2 para SQL - más módulos para 3B
    "target_modules": [
        "q_proj", "k_proj", "v_proj"
    ]
}

# Crear directorios
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["logs_dir"], exist_ok=True)

print("✅ Configuración establecida")
print(f"📦 Modelo: {CONFIG['model_name']}")
print(f"📄 CSV: {CONFIG['csv_file_path']}")
print(f"📊 Max muestras: {CONFIG['num_samples']}")
print(f"🎯 LoRA rank: {LORA_CONFIG['r']}")
print(f"📁 Output: {CONFIG['output_dir']}")

## 📋 Configuración Específica para Kaggle con Dataset CSV

### ⚙️ Configuración óptima para Kaggle:
- **Modelo**: Llama 3.2 3B (más potente que 1B)
- **Dataset**: CSV estratificado pre-limpiado (sin filtros adicionales)
- **GPU**: Aprovecha T4/P100 de Kaggle
- **Memoria**: Optimizada para ~16GB RAM
- **Batch size**: 1 con gradient_accumulation=8 (simula batch=8)
- **Secuencias**: 512 tokens (más contexto)
- **LoRA rank**: 16 (más parámetros entrenables)

### 🚀 Tiempo estimado en Kaggle:
- **Con GPU T4**: 2-4 horas
- **Con GPU P100**: 1.5-3 horas
- **Con CPU**: 8-12 horas (no recomendado)

### 💡 Ventajas del CSV pre-procesado:
1. **Sin limpieza**: Datos ya filtrados y balanceados
2. **Estratificado**: Distribución equilibrada de complejidad
3. **Metadatos**: Información adicional (dominio, complejidad, tipo)
4. **Más rápido**: Carga directa sin procesamiento

## 5. Carga del Dataset CSV

In [ ]:
def cargar_datos_csv():
    """Carga datos desde el CSV estratificado pre-procesado"""
    print("📥 Cargando dataset desde CSV estratificado...")
    
    # Verificar que el archivo existe
    if not os.path.exists(CONFIG["csv_file_path"]):
        raise FileNotFoundError(f"No se encontró el archivo CSV: {CONFIG['csv_file_path']}")
    
    # Cargar CSV
    df = pd.read_csv(CONFIG["csv_file_path"])
    
    print(f"📦 Dataset cargado: {len(df)} registros totales")
    print(f"📝 Columnas disponibles: {list(df.columns)}")
    
    # Verificar columnas necesarias
    required_columns = ['sql_prompt', 'sql_context', 'sql']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas requeridas: {missing_columns}")
    
    # Filtrar solo registros de entrenamiento (si existe columna split)
    if 'split' in df.columns:
        train_df = df[df['split'] == 'train'].copy()
        print(f"🚂 Registros de entrenamiento: {len(train_df)}")
    else:
        train_df = df.copy()
        print(f"📊 Usando todos los registros para entrenamiento: {len(train_df)}")
    
    # NO HACER LIMPIEZA - el CSV ya está pre-procesado
    print("✅ Usando dataset pre-limpiado, sin filtros adicionales")
    
    # Limitar muestras si es necesario
    if len(train_df) > CONFIG["num_samples"]:
        print(f"✂️  Limitando a {CONFIG['num_samples']} muestras")
        # Mantener distribución estratificada si existe columna de complejidad
        if 'sql_complexity' in train_df.columns:
            train_df = train_df.groupby('sql_complexity').head(CONFIG['num_samples']//train_df['sql_complexity'].nunique()).reset_index(drop=True)
        else:
            train_df = train_df.head(CONFIG["num_samples"]).reset_index(drop=True)
    
    print(f"✅ Dataset final: {len(train_df)} ejemplos")
    
    # Mostrar estadísticas del dataset
    if 'sql_complexity' in train_df.columns:
        print(f"📈 Distribución por complejidad:")
        complexity_counts = train_df['sql_complexity'].value_counts()
        for complexity, count in complexity_counts.items():
            print(f"   {complexity}: {count} ({count/len(train_df)*100:.1f}%)")
    
    if 'domain' in train_df.columns:
        print(f"🏷️  Dominios únicos: {train_df['domain'].nunique()}")
    
    if 'sql_task_type' in train_df.columns:
        print(f"🎯 Tipos de tarea únicos: {train_df['sql_task_type'].nunique()}")
    
    # Mostrar estadísticas de longitud
    print(f"\n📊 Estadísticas de longitud:")
    print(f"   SQL promedio: {train_df['sql'].str.len().mean():.0f} caracteres")
    print(f"   Pregunta promedio: {train_df['sql_prompt'].str.len().mean():.0f} caracteres")
    print(f"   Contexto promedio: {train_df['sql_context'].str.len().mean():.0f} caracteres")
    
    # Dividir train_df en 80% train y 20% eval
    from sklearn.model_selection import train_test_split
    train_df_80, eval_df_20 = train_test_split(train_df, test_size=0.2, random_state=42)
    train_df_80 = train_df_80.reset_index(drop=True)
    eval_df_20 = eval_df_20.reset_index(drop=True)
    
    print(f"\n📊 División para evaluación:")
    print(f"   Train (80%): {len(train_df_80)} ejemplos")
    print(f"   Eval (20%): {len(eval_df_20)} ejemplos")
    
    return train_df_80, eval_df_20

# Cargar datos
train_df, eval_df = cargar_datos_csv()

# Mostrar ejemplo
print(f"\n📝 Ejemplo del dataset:")
ejemplo = train_df.iloc[0]
print(f"Dominio: {ejemplo.get('domain', 'N/A')}")
print(f"Complejidad: {ejemplo.get('sql_complexity', 'N/A')}")
print(f"Pregunta: {ejemplo['sql_prompt'][:100]}...")
print(f"SQL: {ejemplo['sql']}")
if 'sql_explanation' in ejemplo:
    print(f"Explicación: {ejemplo['sql_explanation'][:100]}...")

## 6. Formateo de Datos para Llama 3.2

In [ ]:
def formatear_para_llama32(df):
    """Formatea los datos usando el template de Llama 3.2"""
    print("🔄 Formateando datos para Llama 3.2...")
    
    # Template optimizado para text-to-SQL
    TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{sql}<|eot_id|>"""
    
    formatted_data = []
    
    for _, row in df.iterrows():
        # Crear texto formateado usando las columnas del CSV
        text = TEMPLATE.format(
            schema=row['sql_context'].strip(),
            question=row['sql_prompt'].strip(),
            sql=row['sql'].strip()
        )
        
        formatted_data.append({"text": text})
    
    print(f"✅ {len(formatted_data)} ejemplos formateados")
    
    # Mostrar ejemplo formateado
    print(f"\n📝 Ejemplo formateado:")
    print(formatted_data[0]["text"][:500] + "...")
    
    return formatted_data

# Formatear datos
training_data = formatear_para_llama32(train_df)
eval_data = formatear_para_llama32(eval_df)

# Crear datasets
train_dataset = Dataset.from_list(training_data)
eval_dataset = Dataset.from_list(eval_data)
print(f"\n📦 Datasets creados:")
print(f"   Train: {len(train_dataset)} ejemplos")
print(f"   Eval: {len(eval_dataset)} ejemplos")

## 7. Carga del Modelo Llama 3.2 3B

In [ ]:
def cargar_modelo_llama32():
    """Carga Llama 3.2 3B con configuración optimizada"""
    print(f"🤖 Cargando {CONFIG['model_name']}...")
    
    # Cargar tokenizador
    print("📝 Cargando tokenizador...")
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["model_name"]
    )
    # Configurar pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    print(f"   ✅ Tokenizador cargado. Vocab: {len(tokenizer)}")
    
    # Cargar modelo con configuración optimizada para Kaggle
    print("🧠 Cargando modelo base...")
    
    # Configuración para Kaggle - usar GPU si disponible
    device_map = "auto" if torch.cuda.is_available() else None
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=True,
    )
    
    # Preparar para LoRA si no hay quantización
    if not torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    
    print(f"   ✅ Modelo cargado")
    print(f"   💾 Parámetros: {model.num_parameters():,}")
    
    return model, tokenizer

# Cargar modelo
base_model, tokenizer = cargar_modelo_llama32()

## 8. Aplicación de LoRA

In [ ]:
def aplicar_lora(model):
    """Aplica LoRA al modelo base"""
    print("🔧 Aplicando LoRA...")
    
    # Crear configuración LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=LORA_CONFIG["target_modules"],
    )
    
    # Aplicar LoRA
    model_lora = get_peft_model(model, lora_config)

    # Estadísticas
    trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_lora.parameters())
    
    print(f"✅ LoRA aplicado")
    print(f"📊 Parámetros entrenables: {trainable:,} ({trainable/total*100:.2f}%)")
    print(f"📊 Parámetros totales: {total:,}")
    
    # Mostrar módulos LoRA
    print(f"\n🎯 Módulos LoRA activos:")
    lora_modules = [name for name, _ in model_lora.named_modules() if "lora" in name.lower()]
    for module in lora_modules[:5]:  # Mostrar solo los primeros 5
        print(f"   {module}")
    if len(lora_modules) > 5:
        print(f"   ... y {len(lora_modules)-5} más")
    
    return model_lora

# Aplicar LoRA
model = aplicar_lora(base_model)

## 9. Configuración del Entrenamiento

In [ ]:
def crear_training_arguments():
    """Crea argumentos de entrenamiento optimizados"""
    print("⚙️ Configurando entrenamiento...")
    
    # Calcular pasos
    num_samples = len(train_dataset)
    num_eval_samples = len(eval_dataset)
    effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
    steps_per_epoch = num_samples // effective_batch_size
    max_steps = steps_per_epoch * CONFIG["num_epochs"]
    warmup_steps = int(max_steps * CONFIG["warmup_ratio"])
    
    print(f"📊 Configuración de entrenamiento:")
    print(f"   Train samples: {num_samples}")
    print(f"   Eval samples: {num_eval_samples}")
    print(f"   Batch efectivo: {effective_batch_size}")
    print(f"   Pasos por época: {steps_per_epoch}")
    print(f"   Pasos totales: {max_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    print(f"   Eval cada: {CONFIG['eval_steps']} pasos")
    print(f"   Max sequence length: {CONFIG['max_seq_length']} (controlado por dataset)")
    
    training_args = TrainingArguments(
        # Directorios
        output_dir=CONFIG["output_dir"],
        logging_dir=CONFIG["logs_dir"],
        
        # Entrenamiento
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        learning_rate=CONFIG["learning_rate"],
        
        # Scheduler
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        
        # Guardado y evaluación
        save_steps=CONFIG["save_steps"],
        save_total_limit=2,
        logging_steps=CONFIG["logging_steps"],
        evaluation_strategy=CONFIG["eval_strategy"],
        eval_steps=CONFIG["eval_steps"],
        load_best_model_at_end=CONFIG["load_best_model_at_end"],
        metric_for_best_model=CONFIG["metric_for_best_model"],
        
        # Optimización para Kaggle
        optim="adamw_8bit",      # Optimizador más eficiente
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        # Precisión - usar fp16 si hay GPU
        fp16=torch.cuda.is_available(),
        bf16=False,
        
        # Otros
        dataloader_drop_last=True,
        remove_unused_columns=False,
        report_to="none",  # Sin logging externo
        seed=42,
    )
    
    return training_args

# Crear argumentos
training_arguments = crear_training_arguments()
print("✅ Argumentos de entrenamiento creados")

## 10. Preparación del Trainer

In [ ]:
def crear_trainer():
    """Crea el SFTTrainer"""
    print("🏃‍♂️ Preparando SFTTrainer...")
    
    # Configuración básica y mínima compatible con trl 0.7.11
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_arguments,
    )
    
    # Configurar tokenizer manualmente
    trainer.tokenizer = tokenizer
    if trainer.tokenizer.pad_token is None:
        trainer.tokenizer.pad_token = trainer.tokenizer.eos_token
    
    print("✅ SFTTrainer preparado con configuración básica")
    print(f"📦 Train dataset: {len(trainer.train_dataset)} ejemplos")
    print(f"📦 Eval dataset: {len(trainer.eval_dataset)} ejemplos")
    print(f"🔤 Tokenizer configurado manualmente")
    
    return trainer

# Crear trainer
trainer = crear_trainer()

## 11. ¡ENTRENAMIENTO!

**⚠️ IMPORTANTE:**
- Este proceso puede tomar 1-3 horas
- Monitorea la pérdida (loss) - debe disminuir
- Si hay errores de memoria, reduce `batch_size` o `max_seq_length`
- **Dataset CSV**: Sin filtros adicionales, datos pre-procesados

In [ ]:
def entrenar():
    """Ejecuta el entrenamiento"""
    print("🚀 INICIANDO ENTRENAMIENTO CON DATASET CSV")
    print("=" * 50)
    
    start_time = datetime.now()
    print(f"⏰ Inicio: {start_time.strftime('%H:%M:%S')}")
    print(f"📄 Fuente: Dataset CSV estratificado")
    print(f"📊 Ejemplos: {len(train_dataset)}")
    
    try:
        # ¡ENTRENAR!
        result = trainer.train()
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print("\n🎉 ENTRENAMIENTO COMPLETADO")
        print("=" * 50)
        print(f"⏰ Fin: {end_time.strftime('%H:%M:%S')}")
        print(f"⏱️ Duración: {duration}")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        return True, result
        
    except KeyboardInterrupt:
        print("\n⚠️ Entrenamiento interrumpido")
        return False, None
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return False, None

# ¡EJECUTAR ENTRENAMIENTO!
success, training_result = entrenar()

## 12. Guardar Modelo Entrenado

In [ ]:
def guardar_modelo():
    """Guarda el modelo entrenado"""
    if not success:
        print("❌ No se puede guardar - entrenamiento no completado")
        return None
    
    print("💾 Guardando modelo...")
    
    # Directorio final
    final_dir = f"{CONFIG['output_dir']}/final"
    os.makedirs(final_dir, exist_ok=True)
    
    # Guardar modelo LoRA
    model.save_pretrained(final_dir)
    print(f"✅ Modelo LoRA guardado en: {final_dir}")
    
    # Guardar tokenizador
    tokenizer.save_pretrained(final_dir)
    print(f"✅ Tokenizador guardado")
    
    # Guardar configuración
    config_info = {
        "base_model": CONFIG["model_name"],
        "dataset_source": "CSV",
        "csv_file": CONFIG["csv_file_path"],
        "num_samples": len(train_dataset),
        "lora_config": LORA_CONFIG,
        "training_config": CONFIG,
        "training_loss": training_result.training_loss if training_result else None,
        "date": datetime.now().isoformat(),
        "dataset_info": {
            "total_examples": len(train_df) + len(eval_df),
            "complexity_distribution": train_df['sql_complexity'].value_counts().to_dict() if 'sql_complexity' in train_df.columns else None,
            "domains": train_df['domain'].nunique() if 'domain' in train_df.columns else None
        }
    }
    
    with open(f"{final_dir}/training_info.json", "w") as f:
        json.dump(config_info, f, indent=2)
    
    print(f"✅ Información guardada")
    print(f"\n📁 Modelo completo en: {final_dir}")
    
    return final_dir

# Guardar modelo
model_path = guardar_modelo()

## 13. Prueba del Modelo

In [ ]:
def probar_modelo():
    """Prueba el modelo entrenado con ejemplos reales del dataset CSV"""
    if not model_path:
        print("❌ No hay modelo para probar")
        return
    
    print("🧪 PROBANDO MODELO ENTRENADO CON DATASET CSV")
    print("=" * 50)
    
    def generar_sql(schema, question):
        """Genera SQL usando el modelo entrenado"""
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        # Tokenizar
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generar
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        # Decodificar
        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        # Extraer SQL generado
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part
    
    # Usar solo ejemplos reales del CSV con split 'train'
    print("🔬 Probando con ejemplos reales del dataset CSV (split='train'):")
    
    # Usar ejemplos de entrenamiento (ya divididos)
    train_examples = train_df
    
    if len(train_examples) == 0:
        print("❌ No se encontraron ejemplos de entrenamiento en el dataset")
        return
    
    # Seleccionar 5-10 ejemplos diversos
    num_tests = min(10, len(train_examples))
    print(f"📊 Seleccionando {num_tests} ejemplos del dataset...")
    
    # Intentar seleccionar ejemplos diversos por complejidad si existe la columna
    if 'sql_complexity' in train_examples.columns:
        # Seleccionar ejemplos de diferentes complejidades
        test_examples = []
        complexities = train_examples['sql_complexity'].unique()
        samples_per_complexity = max(1, num_tests // len(complexities))
        
        for complexity in complexities:
            complexity_examples = train_examples[train_examples['sql_complexity'] == complexity]
            selected = complexity_examples.sample(n=min(samples_per_complexity, len(complexity_examples)), random_state=42)
            test_examples.append(selected)
        
        test_df = pd.concat(test_examples).head(num_tests)
        print(f"✅ Ejemplos seleccionados por complejidad: {test_df['sql_complexity'].value_counts().to_dict()}")
    else:
        # Selección aleatoria si no hay columna de complejidad
        test_df = train_examples.sample(n=num_tests, random_state=42)
        print(f"✅ Ejemplos seleccionados aleatoriamente")
    
    # Probar cada ejemplo
    for i, (_, row) in enumerate(test_df.iterrows(), 1):
        print(f"\n🧪 PRUEBA {i}/{num_tests}:")
        print("-" * 60)
        
        # Mostrar metadatos si están disponibles
        if 'domain' in row:
            print(f"🏷️  Dominio: {row['domain']}")
        if 'sql_complexity' in row:
            print(f"⚡ Complejidad: {row['sql_complexity']}")
        if 'sql_task_type' in row:
            print(f"🎯 Tipo de tarea: {row['sql_task_type']}")
        
        # Mostrar información del test
        schema = row['sql_context']
        question = row['sql_prompt']
        expected_sql = row['sql']
        
        print(f"🏗️  Schema: {schema[:100]}{'...' if len(schema) > 100 else ''}")
        print(f"❓ Pregunta: {question}")
        print(f"✅ SQL Esperado: {expected_sql}")
        
        try:
            generated_sql = generar_sql(schema, question)
            print(f"🤖 SQL Generado: {generated_sql}")
            
            # Análisis básico de similitud
            if generated_sql.lower() == expected_sql.lower():
                print("🎯 Estado: ✅ EXACTO")
            elif generated_sql.lower() in expected_sql.lower() or expected_sql.lower() in generated_sql.lower():
                print("🎯 Estado: ✅ SIMILAR")
            elif any(keyword in generated_sql.upper() for keyword in ['SELECT', 'FROM', 'WHERE', 'INSERT', 'UPDATE', 'DELETE']):
                print("🎯 Estado: ⚠️  SQL VÁLIDO")
            else:
                print("🎯 Estado: ❌ PROBLEMÁTICO")
                
            # Mostrar explicación si está disponible
            if 'sql_explanation' in row and pd.notna(row['sql_explanation']):
                print(f"💡 Explicación: {row['sql_explanation'][:150]}{'...' if len(row['sql_explanation']) > 150 else ''}")
                
        except Exception as e:
            print(f"❌ Error generando SQL: {e}")
            print("🎯 Estado: ❌ ERROR")
        
        print("-" * 60)
    
    print(f"\n🏁 PRUEBAS COMPLETADAS")
    print(f"📊 Total de ejemplos probados: {len(test_df)}")
    if 'sql_complexity' in test_df.columns:
        print(f"📈 Distribución por complejidad:")
        for complexity, count in test_df['sql_complexity'].value_counts().items():
            print(f"   {complexity}: {count} ejemplos")

# Probar modelo
probar_modelo()

## 14. Script de Integración

In [ ]:
def crear_script_uso():
    """Crea script para usar el modelo en tu proyecto"""
    if not model_path:
        print("❌ No hay modelo para crear script")
        return
    
    script = f'''# Script para usar el modelo SQL Llama 3.2 entrenado con CSV
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

class LlamaSQLGeneratorCSV:
    def __init__(self, model_path="{model_path}"):
        print("🤖 Cargando modelo SQL Llama 3.2 (entrenado con CSV)...")
        
        # Cargar modelo base
        self.base_model = AutoModelForCausalLM.from_pretrained(
            "{CONFIG['model_name']}",
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        
        # Cargar adaptadores LoRA
        self.model = PeftModel.from_pretrained(self.base_model, model_path)
        
        # Cargar tokenizador
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        print("✅ Modelo cargado (entrenado con dataset CSV estratificado)")
    
    def generar_sql(self, schema, question):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{{schema}}

Question: {{question}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {{k: v.to(self.model.device) for k, v in inputs.items()}}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part

# Función compatible con tu código existente
def generar_sql_con_llama_csv(prompt):
    generator = LlamaSQLGeneratorCSV()
    # Parsear prompt simple
    if "Schema:" in prompt and "Question:" in prompt:
        schema = prompt.split("Question:")[0].replace("Schema:", "").strip()
        question = prompt.split("Question:")[1].replace("Return only the SQL query:", "").strip()
    else:
        schema = "Unknown"
        question = prompt
    
    return generator.generar_sql(schema, question)

# Ejemplo de uso
if __name__ == "__main__":
    generator = LlamaSQLGeneratorCSV()
    sql = generator.generar_sql(
        "CREATE TABLE users (id INT, name VARCHAR(50), age INT);",
        "Get users older than 25"
    )
    print(f"SQL: {{sql}}")
'''
    
    # Guardar script
    script_path = "../scripts/llama_sql_generator_csv.py"
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script)
    
    # Instrucciones
    instructions = f'''# CÓMO USAR TU MODELO LLAMA SQL (ENTRENADO CON CSV)

## En tu run_batch.py:
```python
# Cambiar:
from scripts.generate_sql import generar_sql_con_ollama
# Por:
from scripts.llama_sql_generator_csv import generar_sql_con_llama_csv

# Y usar:
sql_generado = generar_sql_con_llama_csv(prompt)
```

## Uso directo:
```python
from scripts.llama_sql_generator_csv import LlamaSQLGeneratorCSV

generator = LlamaSQLGeneratorCSV()
sql = generator.generar_sql(schema, question)
```

## Modelo entrenado con:
- **Dataset**: CSV estratificado ({CONFIG["csv_file_path"]})
- **Ejemplos**: {len(train_dataset)} muestras
- **Modelo base**: {CONFIG["model_name"]}
- **LoRA**: Rank {LORA_CONFIG["r"]}, Alpha {LORA_CONFIG["lora_alpha"]}

## Modelo guardado en: {model_path}

## Ventajas del modelo CSV:
1. **Datos estratificados**: Distribución equilibrada de complejidad
2. **Sin limpieza**: Dataset pre-procesado y validado
3. **Metadatos**: Información de dominio y tipo de tarea
4. **Calidad**: Ejemplos curados manualmente
'''
    
    with open("../COMO_USAR_LLAMA_SQL_CSV.md", "w", encoding="utf-8") as f:
        f.write(instructions)
    
    print(f"✅ Script creado: {script_path}")
    print(f"✅ Instrucciones: ../COMO_USAR_LLAMA_SQL_CSV.md")

# Crear scripts
crear_script_uso()

## 🎉 ¡ENTRENAMIENTO COMPLETADO CON DATASET CSV!

### ✅ Lo que has logrado:

1. **Modelo entrenado**: Llama 3.2 3B especializado en SQL con datos CSV
2. **LoRA aplicado**: Entrenamiento eficiente (~1% parámetros)
3. **Datos de calidad**: Dataset estratificado pre-procesado
4. **Sin limpieza**: Datos ya validados y balanceados
5. **Modelo guardado**: Listo para usar en producción
6. **Scripts creados**: Integración fácil

### 📁 Archivos generados:

- `../models/llama-sql-lora-csv/final/` - Modelo entrenado
- `../scripts/llama_sql_generator_csv.py` - Script de uso
- `../COMO_USAR_LLAMA_SQL_CSV.md` - Instrucciones

### 🚀 Ventajas del modelo CSV:

1. **Estratificación**: Distribución equilibrada de complejidad SQL
2. **Metadatos**: Información de dominio, tipo de tarea, explicaciones
3. **Calidad**: Datos curados sin necesidad de limpieza adicional
4. **Eficiencia**: Carga más rápida y entrenamiento directo

### 🎯 Próximos pasos:

1. **Probar más ejemplos** en la celda anterior
2. **Integrar en run_batch.py** usando el script generado
3. **Comparar rendimiento** vs modelo original
4. **Evaluar con datos de test** del CSV

¡Tu modelo Llama 3.2 SQL entrenado con dataset CSV está listo! 🎯✨